In [113]:
#그래프관련 import
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
# matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus']=False

import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso,LogisticRegression
from sklearn.preprocessing import StandardScaler,PolynomialFeatures,OneHotEncoder
from sklearn.metrics import mean_absolute_error
from scipy.special import expit,softmax # expit : 이진분류함수 , softmax : 다중분류 0-1 숫자로 변환
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier


In [114]:
fish = pd.read_csv('data/fish_logistic.csv')
fish.head()

,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.5200,4.0200
1,Bream,290.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,26.5,31.1,12.3778,4.6961
3,Bream,363.0,29.0,33.5,12.7300,4.4555
4,Bream,430.0,29.0,34.0,12.4440,5.1340


In [115]:
# 로지스틱 회귀 - 다중분류 모델을 이용해서 생선을 분류하시오.
# 다중 : 소프트맥스 사용해서 퍼센트를 구하시오.
# 무게,길이,대각선,높이,넓이를 입력해서 생선을 분류해서 출력하시오.
data = fish.iloc[:,1:].values
label = fish.iloc[:,0].values


In [116]:
# fish.info()
fish.shape
fish.describe()

,Weight,Length,Diagonal,Height,Width
count,159.000000,159.000000,159.000000,159.000000,159.000000
mean,398.326415,28.415723,31.227044,8.970994,4.417486
std,357.978317,10.716328,11.610246,4.286208,1.685804
min,0.000000,8.400000,8.800000,1.728400,1.047600
25%,120.000000,21.000000,23.150000,5.944800,3.385650
50%,273.000000,27.300000,29.400000,7.786000,4.248500
75%,650.000000,35.500000,39.650000,12.365900,5.584500
max,1650.000000,63.400000,68.000000,18.957000,8.142000


In [117]:
train_data,test_data,train_label,test_label = train_test_split(data,label,random_state=42,test_size=0.2,stratify=label)

In [118]:
ss = StandardScaler()
ss.fit(train_data)
train_scaled = ss.transform(train_data)
test_scaled = ss.transform(test_data)

In [119]:
param = {
    'C' :[0.01,0.1,1,10,100,1000],
    'max_iter' : [1000,2000]
}

In [ ]:
gs = GridSearchCV(LogisticRegression(random_state=42),param,return_train_score=True,n_jobs=-1)
gs.fit(train_scaled,train_label)
lr = gs.best_estimator_
print('train정확도 : ',lr.score(train_scaled,train_label))
print('test정확도 : ',lr.score(test_scaled,test_label))

print('가장좋은 파라미터값 : ',gs.best_params_)
print(gs.cv_results_['mean_train_score'])
print(gs.cv_results_['mean_test_score'])

# train정확도 :  0.9921259842519685
# test정확도 :  0.96875
# 가장좋은 파라미터값 :  {'C': 1000, 'max_iter': 1000}

train정확도 :  0.9921259842519685
test정확도 :  0.96875
가장좋은 파라미터값 :  {'C': 1000, 'max_iter': 1000}
[0.45859056 0.45859056 0.71258008 0.71258008 0.81688992 0.81688992
 0.91143467 0.91143467 0.98033392 0.98033392 0.99407882 0.99407882]
[0.45723077 0.45723077 0.70892308 0.70892308 0.81846154 0.81846154
 0.88953846 0.88953846 0.91292308 0.91292308 0.92123077 0.92123077]


In [121]:
predict = lr.predict(test_scaled[:5])
predict

array(['Bream', 'Roach', 'Smelt', 'Perch', 'Perch'], dtype=object)

In [122]:
decision = lr.decision_function(test_scaled[:5])
soft1 = softmax(decision,axis=1)
print(lr.classes_)
np.round(soft1,decimals=3)

['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']


array([[1.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.993, 0.   , 0.007],
       [0.   , 0.   , 0.001, 0.   , 0.   , 0.999, 0.   ],
       [0.   , 0.   , 0.997, 0.   , 0.   , 0.   , 0.003],
       [0.   , 0.002, 0.863, 0.   , 0.105, 0.   , 0.03 ]])

In [123]:
fish.head()
fish.tail()

,Species,Weight,Length,Diagonal,Height,Width
154,Smelt,12.2,12.2,13.4,2.0904,1.3936
155,Smelt,13.4,12.4,13.5,2.4300,1.2690
156,Smelt,12.2,13.0,13.8,2.2770,1.2558
157,Smelt,19.7,14.3,15.2,2.8728,2.0672
158,Smelt,19.9,15.0,16.2,2.9322,1.8792


In [124]:
# 12.2,12.2,13.4,2.09,1.39
# 242.0,25.4,30.0,11.52,4.02
# input1 = '242.0,25.4,30.0,11.52,4.02'
input1 = input('무게,길이,대각선,높이,넓이를 입력하시오. 예)242.0,25.4,30.0,11.52,4.02')
input2 = input1.split(',')
input3 = list(map(float,input2))

input_scaled = ss.transform([input3])
predict = lr.predict(input_scaled)
print('예측생선 : ',predict[0])

예측생선 :  Bream


In [125]:
from sklearn.pipeline import make_pipeline
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1000,max_iter=1000)
)

In [126]:
model.fit(train_data,train_label)

,steps,"[('standardscaler', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000


In [127]:
print('train정확도 : ',model.score(train_data,train_label))
print('test정확도 : ',model.score(test_data,test_label))

train정확도 :  0.9921259842519685
test정확도 :  0.96875


In [128]:
import joblib
joblib.dump(model,'model_lr.joblib')
print('저장완료')


저장완료
